In [ ]:
from pathlib import Path

import numpy as np
import open3d as o3d

from plant_shape_analysis import PlantSequencesDataset, LeafSequencesDataset
from plant_shape_analysis.vis.plot_functions import visualize_plant_sequence, visualize_leaf_sequence, visualize_point_cloud   

%load_ext autoreload
%autoreload 2

In [ ]:
selected_sequences = [
"maize_control_plant1",
"maize_control_plant2",
"maize_control_plant3",
"sorghum_control_plant1",
"sorghum_control_plant2",
"sorghum_highlight_plant2",
"tobacco_control_plant1",
"tobacco_control_plant2",
"tobacco_shade_plant3",
"tomato1_heat_plant3",
"tomato1_shade_plant1",
"tomato2_control_plant3"
]

## Compare aligned vs unaligned

In [ ]:
aligned_leaf_dataset = LeafSequencesDataset(
    Path("../data/TrackPlant3D/versions"),
    use_ply=True,
    version="v2",
    apply_alignment=True, # True to test leaf alignment and debug normals
    auto_download=False,
    min_timepoints=5,

)
unaligned_leaf_dataset = LeafSequencesDataset(
    Path("../data/TrackPlant3D/versions"),
    use_ply=True,
    version="v2",
    apply_alignment=False, 
    auto_download=False,
    min_timepoints=5,
)

In [ ]:
# Visualize all leaves of selected sequences
selected_leaves = []
for plant_name in sorted(selected_sequences):
    all_leaves = aligned_leaf_dataset.get_timeseries_by_plant_sequence(plant_name)
    selected_leaves.extend([leaf["sequence_name"] for leaf in all_leaves])
len(selected_leaves) # 44 leaf sequences to inspect

In [ ]:
# Get leaf name
i = 12
leaf_name = selected_leaves[i]
leaf_name

In [ ]:
# Plot aligned leaf sequence
leaf_sample = aligned_leaf_dataset.get_timeseries_by_sequence_name(leaf_name)
visualize_leaf_sequence(leaf_sample, window_name=f"{leaf_sample['sequence_name']}", show_coordinate_frame=True, show_connections=True)

# Plot aligned leaf sequence
leaf_sample = unaligned_leaf_dataset.get_timeseries_by_sequence_name(leaf_name)
visualize_leaf_sequence(leaf_sample, window_name=f"{leaf_sample['sequence_name']}", show_coordinate_frame=True, show_connections=True)

In [ ]:
# # Visualize all leaves of selected sequences
# for plant_name in selected_sequences:
#     all_leaves = aligned_leaf_dataset.get_timeseries_by_plant_sequence(plant_name)
#     for leaf_seq in all_leaves:
#         visualize_leaf_sequence(leaf_seq, window_name=f"{leaf_seq['sequence_name']}", show_coordinate_frame=True, show_connections=True)

## Visualize alignment results

In [ ]:
leaf_dataset = LeafSequencesDataset(
    Path("../data/TrackPlant3D/versions"),
    use_ply=True,
    version="v2",
    apply_alignment=True, # True to test leaf alignment and debug normals
    auto_download=False,
    min_timepoints=5,
)

In [ ]:
# Visualize all leaves of selected sequences
for plant_name in selected_sequences:
    all_leaves = leaf_dataset.get_timeseries_by_plant_sequence(plant_name)
    for leaf_seq in all_leaves:
        visualize_leaf_sequence(leaf_seq, window_name=f"{leaf_seq['sequence_name']}", show_coordinate_frame=True, show_connections=True, show_leaf_tips=True, show_principal_axes=True)

## Improve alignment pipeline

### Visualize aligned leaf sequence

In [ ]:
leaf_dataset = LeafSequencesDataset(
    Path("../data/TrackPlant3D/versions"),
    use_ply=True,
    version="v2",
    apply_alignment=True, # True to test leaf alignment and debug normals
    auto_download=False,
    min_timepoints=5,
)

In [ ]:
leaf_dataset.print_alignment_summary()

In [ ]:
# Visualize all leaves of selected sequences
for plant_name in reversed(selected_sequences):
    all_leaves = leaf_dataset.get_timeseries_by_plant_sequence(plant_name)
    for leaf_seq in all_leaves:
        visualize_leaf_sequence(leaf_seq, window_name=f"{leaf_seq['sequence_name']}", show_coordinate_frame=True, show_connections=True)

### Refine leaf alignment pipeline

In [ ]:
from plant_shape_analysis.alignment import align_main_axis_to_z, compute_pca_basis

In [ ]:
leaf_dataset = LeafSequencesDataset(
    Path("../data/TrackPlant3D/versions"),
    use_ply=True,
    version="v2",
    apply_alignment=False, # True to test leaf alignment and debug normals
    auto_download=False,
    min_timepoints=5,
)

In [ ]:
seq_name = "tomato2_control_plant3_leaf3"
test_seq = leaf_dataset.get_timeseries_by_sequence_name(seq_name)
plant = leaf_dataset.plant_dataset.get_timeseries_by_sequence_name(seq_name.split("_leaf")[0])

In [ ]:
# Visualize unaligned leaf sequence
visualize_leaf_sequence(test_seq, window_name=f"{test_seq['sequence_name']}", show_coordinate_frame=True, show_connections=True)

Use stems points to get rough insertion point and use it to correct sign in PCA main axis direction

In [ ]:
plant = leaf_dataset.plant_dataset.get_timeseries_by_sequence_name(seq_name.split("_leaf")[0])
stem_timepoints = []
for timepoint in plant['timepoints']:
    stem_mask = timepoint["labels"] == 0
    stem_points = timepoint["points"][stem_mask]
    stem_normals = timepoint["normals"][stem_mask] if "normals" in timepoint else None
    stem_timepoints.append({
        "day": timepoint["day"],
        "points": stem_points,
        "normals": stem_normals,
        "leaf_tip": None,
    })
visualize_leaf_sequence({"timepoints": stem_timepoints, "sequence_name": plant["sequence_name"], "leaf_id": None}, show_coordinate_frame=True, show_leaf_tips=False, show_normals=False)

In [ ]:
def translate_to_origin(points):
    """Translate points so their centroid is at the origin."""
    centroid = np.mean(points, axis=0)
    return points - centroid, centroid

In [ ]:
timepoints_before = []
timepoints_centered = []
for data in test_seq['timepoints']:
    tp = {
        "day": data["day"],
        "points": data["points"].copy(),
        "normals": data.get("normals").copy() if data.get("normals") is not None else None,
        "leaf_tip": data.get("leaf_tip").copy() if data.get("leaf_tip") is not None else None,
    }
    timepoints_before.append(tp)

    # Center points to origin
    points_centered, centroid = translate_to_origin(data['points'].copy())
    tip_centered = tp['leaf_tip'] - centroid if tp.get('leaf_tip') is not None else None

    timepoints_centered.append({
        "day": data["day"],
        "points": points_centered,
        "normals": tp["normals"],
        "leaf_tip": tip_centered
    })


print(f"Testing sequence: {seq_name}")
print(f"Number of timepoints: {len(timepoints_before)}")
print(f"Has tips: {[tp['leaf_tip'] is not None for tp in timepoints_before]}")

In [ ]:
# Visualize centered unaligned leaf sequence
visualize_leaf_sequence({"timepoints": timepoints_centered, "sequence_name": seq_name, "leaf_id": test_seq["leaf_id"]}, window_name=f"{seq_name}", show_coordinate_frame=True, show_connections=True, spacing=0.)

#### Debug pipeline visually

In [ ]:
# Visually inspect a specific day
test_day_idx = 0  # Change this to test different days
# timepoint_data = test_seq['timepoints'][test_day_idx]
# tp_before = timepoint_data.copy()
leaf_timepoint_data = timepoints_before[test_day_idx]
stem_timepoint_data = stem_timepoints[test_day_idx]
tp_before = leaf_timepoint_data.copy()
stem_data = stem_timepoint_data.copy()

print(f"Testing: {seq_name}, Day {tp_before['day']}")
print(f"Has tip: {tp_before['leaf_tip'] is not None}")
if tp_before['leaf_tip'] is not None:
    print(f"Tip position: {tp_before['leaf_tip']}")

# Compute PCA axis and other debug info
points = tp_before["points"]
center = np.mean(points, axis=0)
centered = points - center
pca_axes = compute_pca_basis(centered)
pca_axis = pca_axes[0] / np.linalg.norm(pca_axes[0])

# Find closest stem point to leaf centroid
distances_to_stem = np.linalg.norm(stem_data['points'] - center, axis=1)
closest_stem_point = stem_data['points'][np.argmin(distances_to_stem)]
print(f"Closest stem point to leaf centroid: {closest_stem_point}")

# Find closest leaf point to the closest stem point
distances_to_leaf = np.linalg.norm(points - closest_stem_point, axis=1)
closest_leaf_point = points[np.argmin(distances_to_leaf)]
print(f"Closest leaf point to stem point: {closest_leaf_point}")

# Project the closest_leaf_point (base) onto the PCA axis
base_centered = closest_leaf_point - center
base_projection = np.dot(base_centered, pca_axis)
print(f"Base projection along PCA axis: {base_projection:.3f}")

# Determine growth direction: we want PCA axis to point from base → tip
# If base has positive projection, flip the axis so base has negative projection
if base_projection > 0:
    growth_direction = -pca_axis
    print("Flipping PCA axis (base was at positive end)")
else:
    growth_direction = pca_axis
    print("Keeping PCA axis (base already at negative end)")

print(f"Growth direction (base→tip): {growth_direction}")

# Verify: check tip projection if available
if tp_before.get('leaf_tip') is not None:
    tip_centered = tp_before['leaf_tip'] - center
    tip_projection = np.dot(tip_centered, growth_direction)
    base_projection_corrected = np.dot(base_centered, growth_direction)
    print(f"\nVerification:")
    print(f"  Base projection along growth_direction: {base_projection_corrected:.3f} (should be negative)")
    print(f"  Tip projection along growth_direction: {tip_projection:.3f} (should be positive)")
    print(f"  Correctly oriented: {base_projection_corrected < 0 and tip_projection > 0}")

pca_axes[0] = growth_direction  # Use computed growth direction for first axis


# Create point cloud
geometries = []
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
geometries.append(pcd)

# Add coordinate frame at origin (world frame)
world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=2.0, origin=[0, 0, 0])
geometries.append(world_frame)

# Add PCA coordinate frame at centroid
# pca_axis is a 3x3 matrix where each row is a principal component
arrow_origin = points.mean(axis=0)
axis_colors = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]  # RGB for PC1, PC2, PC3
axis_lengths = [2, 2, 2]  # Different lengths to distinguish them

for j in range(3):
    axis_vec = pca_axes[j]  # Row j is the j-th principal component
    length = axis_lengths[j]  # Scale for visualization

    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=0.1,
        cone_radius=0.2,
        cylinder_height=length * 0.8,
        cone_height=length * 0.2,
    )

    # Align arrow with axis_vec using Rodrigues' rotation
    z_axis = np.array([0, 0, 1])
    axis_vec_norm = axis_vec / np.linalg.norm(axis_vec)
    v = np.cross(z_axis, axis_vec_norm)
    c = np.dot(z_axis, axis_vec_norm)

    if np.linalg.norm(v) < 1e-8:
        # Already aligned or opposite
        if c > 0:
            R = np.eye(3)
        else:
            R = np.array([[1, 0, 0], [0, -1, 0], [0, 0, -1]])
    else:
        vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        R = np.eye(3) + vx + vx @ vx * ((1 - c) / (np.linalg.norm(v) ** 2))

    arrow.rotate(R, center=np.zeros(3))
    arrow.translate(arrow_origin)
    arrow.paint_uniform_color(axis_colors[j])
    geometries.append(arrow)

# Add rough insertion point
insertion_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.5)
insertion_sphere.translate(closest_leaf_point)
insertion_sphere.paint_uniform_color([1, 1, 0])  # Yellow for insertion
geometries.append(insertion_sphere)


# Add leaf tip sphere if available
if tp_before['leaf_tip'] is not None:
    tip_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.5)
    tip_sphere.translate(tp_before['leaf_tip'])
    tip_sphere.paint_uniform_color([0, 1, 0])  # Green for tip
    geometries.append(tip_sphere)

print("\n=== Legend ===")
print("- Large RGB frame at origin = World coordinate frame")
print("- Small RGB frame = Leaf centroid")
print("- Magenta line = PCA axis")
print("- Red sphere = Max projection point")
print("- Blue sphere = Min projection point")
print("- Green sphere = Leaf tip (if available)")
print("- Gray points = Leaf")

o3d.visualization.draw_geometries(
    geometries,
    window_name=f"BEFORE alignment - {seq_name} Day {tp_before['day']}",
    up=[0, 0, 1],
    front=[1, 0, 0],
    lookat=center,
    zoom=1.5,
)

In [ ]:
# Create point cloud
geometries = []
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
geometries.append(pcd)

# Add coordinate frame at origin (world frame)
world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=2.0, origin=[0, 0, 0])
geometries.append(world_frame)

# Add PCA coordinate frame at centroid
# pca_axis is a 3x3 matrix where each row is a principal component
arrow_origin = points.mean(axis=0)
axis_colors = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]  # RGB for PC1, PC2, PC3
axis_lengths = [2, 2, 2]  # Different lengths to distinguish them

for j in range(3):
    axis_vec = pca_axes[j]  # Row j is the j-th principal component
    length = axis_lengths[j]  # Scale for visualization

    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=0.1,
        cone_radius=0.2,
        cylinder_height=length * 0.8,
        cone_height=length * 0.2,
    )

    # Align arrow with axis_vec using Rodrigues' rotation
    z_axis = np.array([0, 0, 1])
    axis_vec_norm = axis_vec / np.linalg.norm(axis_vec)
    v = np.cross(z_axis, axis_vec_norm)
    c = np.dot(z_axis, axis_vec_norm)

    if np.linalg.norm(v) < 1e-8:
        # Already aligned or opposite
        if c > 0:
            R = np.eye(3)
        else:
            R = np.array([[1, 0, 0], [0, -1, 0], [0, 0, -1]])
    else:
        vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        R = np.eye(3) + vx + vx @ vx * ((1 - c) / (np.linalg.norm(v) ** 2))

    arrow.rotate(R, center=np.zeros(3))
    arrow.translate(arrow_origin)
    arrow.paint_uniform_color(axis_colors[j])
    geometries.append(arrow)

# Add rough insertion point
insertion_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.5)
insertion_sphere.translate(closest_leaf_point)
insertion_sphere.paint_uniform_color([1, 1, 0])  # Yellow for insertion
geometries.append(insertion_sphere)


# Add leaf tip sphere if available
if tp_before['leaf_tip'] is not None:
    tip_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.5)
    tip_sphere.translate(tp_before['leaf_tip'])
    tip_sphere.paint_uniform_color([0, 1, 0])  # Green for tip
    geometries.append(tip_sphere)

print("\n=== Legend ===")
print("- Large RGB frame at origin = World coordinate frame")
print("- Small RGB frame = Leaf centroid")
print("- Magenta line = PCA axis")
print("- Red sphere = Max projection point")
print("- Blue sphere = Min projection point")
print("- Green sphere = Leaf tip (if available)")
print("- Gray points = Leaf")

o3d.visualization.draw_geometries(
    geometries,
    window_name=f"BEFORE alignment - {seq_name} Day {tp_before['day']}",
    up=[0, 0, 1],
    front=[1, 0, 0],
    lookat=center,
    zoom=1.5,
)

#### Batch process

In [ ]:
def align_main_pca_axis_to_growth_direction(leaf_timepoints, stem_timepoints):
    # Apply PCA axis correction to all timepoints
    timepoints_after_flip = []

    # First pass
    for idx, (leaf_timepoint, stem_point) in enumerate(zip(leaf_timepoints, stem_timepoints)):
        # Create a fresh copy of the timepoint
        tp = {
            "day": leaf_timepoint["day"],
            "points": leaf_timepoint["points"].copy(),
            "normals": leaf_timepoint["normals"].copy() if leaf_timepoint.get("normals") is not None else None,
            "leaf_tip": leaf_timepoint["leaf_tip"].copy() if leaf_timepoint.get("leaf_tip") is not None else None,
        }

        points = tp["points"]
        center = np.mean(points, axis=0)
        centered = points - center

        # Compute fresh PCA axes
        pca_axes = compute_pca_basis(centered)
        pca_axis = pca_axes[0] / np.linalg.norm(pca_axes[0])

        # Find closest stem point to leaf centroid (both in original coordinate system)
        distances_to_stem = np.linalg.norm(stem_point['points'] - center, axis=1)
        closest_stem_point = stem_point['points'][np.argmin(distances_to_stem)]

        # Find closest leaf point to the closest stem point (base/insertion)
        distances_to_leaf = np.linalg.norm(points - closest_stem_point, axis=1)
        closest_leaf_point = points[np.argmin(distances_to_leaf)]

        # Project the closest_leaf_point (base) onto the PCA axis
        base_centered = closest_leaf_point - center
        base_projection = np.dot(base_centered, pca_axis)

        # Determine growth direction: we want PCA axis to point from base → tip
        # If base has positive projection, flip the axis so base has negative projection
        if base_projection > 0:
            growth_direction = -pca_axis
            print(f"Day {tp['day']}: Flipping PCA axis (base_proj={base_projection:.3f})")
        else:
            growth_direction = pca_axis
            print(f"Day {tp['day']}: Keeping PCA axis (base_proj={base_projection:.3f})")

        # Store corrected basis
        corrected_basis = np.array([
            growth_direction,
            pca_axes[1] / np.linalg.norm(pca_axes[1]),
            pca_axes[2] / np.linalg.norm(pca_axes[2])
        ])
        tp["basis"] = corrected_basis

        # Verify if tip is available
        if tp.get('leaf_tip') is not None:
            tip_centered = tp['leaf_tip'] - center
            tip_projection = np.dot(tip_centered, growth_direction)
            base_projection_corrected = np.dot(base_centered, growth_direction)
            correctly_oriented = base_projection_corrected < 0 and tip_projection > 0
            print(f"  Verification: base_proj={base_projection_corrected:.3f}, tip_proj={tip_projection:.3f}, correct={correctly_oriented}")
            if not correctly_oriented:
                print("  Warning: PCA axis orientation incorrect despite correction!")
                print(f"  Day {tp['day']}: Flipping PCA axis (base_proj={base_projection_corrected:.3f})")
                corrected_basis[0] = -corrected_basis[0]  # Flip again
        timepoints_after_flip.append(tp)

    # Second pass to ensure consistency (majority vote)

    # Secondary temporal consistency check for timepoints without tips
    print("\n=== Temporal Consistency Check ===")
    timepoints_without_tips = [i for i, tp in enumerate(timepoints_after_flip) if tp["leaf_tip"] is None]

    if len(timepoints_without_tips) > 0 and len(timepoints_without_tips) < len(timepoints_after_flip):
        print(f"Found {len(timepoints_without_tips)} timepoints without tips")

        # Get all growth directions
        growth_directions = np.array([tp["basis"][0] for tp in timepoints_after_flip])

        # Compute pairwise dot products to find majority direction
        # Use timepoints WITH tips as reference (more reliable)
        timepoints_with_tips = [i for i, tp in enumerate(timepoints_after_flip) if tp["leaf_tip"] is not None]

        if len(timepoints_with_tips) > 0:
            # Use average direction of timepoints with tips as reference
            reference_direction = np.mean(growth_directions[timepoints_with_tips], axis=0)
            reference_direction = reference_direction / np.linalg.norm(reference_direction)

            print(f"Using {len(timepoints_with_tips)} timepoints with tips as reference")
        else:
            # Fallback: use majority voting across all timepoints
            print("No tips available, using majority voting")
            # Compute median direction by checking alignment between all pairs
            dot_products = growth_directions @ growth_directions.T
            # For each timepoint, count how many agree with it (dot > 0)
            agreement_counts = (dot_products > 0).sum(axis=1)
            # Use the direction with most agreement as reference
            reference_idx = np.argmax(agreement_counts)
            reference_direction = growth_directions[reference_idx]

        # Check each timepoint without tip
        for idx in timepoints_without_tips:
            tp = timepoints_after_flip[idx]
            current_direction = tp["basis"][0]

            # Check alignment with reference
            dot_product = np.dot(current_direction, reference_direction)

            if dot_product < 0:
                # Pointing opposite direction, flip it
                print(f"  Day {tp['day']}: Flipping axis (dot={dot_product:.3f}, inconsistent with reference). Flipping!")
                tp["basis"][0] = -current_direction
    else:
        print("Skipping temporal consistency check (all or none have tips)")

    print(f"\nProcessed {len(timepoints_after_flip)} timepoints with corrected PCA axes")
    return timepoints_after_flip

timepoints_after_flip = align_main_pca_axis_to_growth_direction(test_seq['timepoints'], stem_timepoints)

visualize_leaf_sequence(
    {"timepoints": timepoints_after_flip, "sequence_name": seq_name, "leaf_id": test_seq["leaf_id"]},
    window_name=f"{seq_name} - Corrected PCA axes",
    show_coordinate_frame=True,
    show_connections=True,
    show_principal_axes=True,
    spacing=30.0
)


# Visualize all leaves of selected sequences
for plant_name in reversed(selected_sequences):
    # Get all leaves
    all_leaves = leaf_dataset.get_timeseries_by_plant_sequence(plant_name)

    # Get stem timepoints
    plant = leaf_dataset.plant_dataset.get_timeseries_by_sequence_name(seq_name.split("_leaf")[0])
    stem_timepoints = []
    for timepoint in plant['timepoints']:
        stem_mask = timepoint["labels"] == 0
        stem_points = timepoint["points"][stem_mask]
        stem_normals = timepoint["normals"][stem_mask] if "normals" in timepoint else None
        stem_timepoints.append({
            "day": timepoint["day"],
            "points": stem_points,
            "normals": stem_normals,
            "leaf_tip": None,
        })
    
    # Visualize all leaves
    for leaf_seq in all_leaves:
        timepoints_after_flip = align_main_pca_axis_to_growth_direction(leaf_seq['timepoints'], stem_timepoints)
        visualize_leaf_sequence(
            {"timepoints": timepoints_after_flip, "sequence_name": leaf_seq['sequence_name'], "leaf_id": leaf_seq["leaf_id"]},
            window_name=f"{leaf_seq['sequence_name']} - Corrected PCA axes",
            show_coordinate_frame=True,
            show_connections=True,
            show_principal_axes=True,
            spacing=30.0
        )

In [ ]:
# Visualize all leaves of selected sequences
for plant_name in reversed(selected_sequences):
    # Get all leaves
    all_leaves = leaf_dataset.get_timeseries_by_plant_sequence(plant_name)

    # Get stem timepoints
    plant = leaf_dataset.plant_dataset.get_timeseries_by_sequence_name(seq_name.split("_leaf")[0])
    stem_timepoints = []
    for timepoint in plant['timepoints']:
        stem_mask = timepoint["labels"] == 0
        stem_points = timepoint["points"][stem_mask]
        stem_normals = timepoint["normals"][stem_mask] if "normals" in timepoint else None
        stem_timepoints.append({
            "day": timepoint["day"],
            "points": stem_points,
            "normals": stem_normals,
            "leaf_tip": None,
        })
    
    # Visualize all leaves
    for leaf_seq in all_leaves:
        timepoints_after_flip = align_main_pca_axis_to_growth_direction(leaf_seq['timepoints'], stem_timepoints)
        visualize_leaf_sequence(
            {"timepoints": timepoints_after_flip, "sequence_name": leaf_seq['sequence_name'], "leaf_id": leaf_seq["leaf_id"]},
            window_name=f"{leaf_seq['sequence_name']} - Corrected PCA axes",
            show_coordinate_frame=True,
            show_connections=True,
            show_principal_axes=True,
            spacing=30.0
        )

#### Apply alignment to z axis

In [ ]:
# Apply base-based alignment
timepoints_after = [tp.copy() for tp in timepoints_before]
stage1_trans = align_main_axis_to_z(timepoints_after, seq_name=seq_name)

In [ ]:
visualize_leaf_sequence({"timepoints": timepoints_after, "sequence_name": seq_name, "leaf_id": test_seq["leaf_id"]}, window_name=f"{seq_name}", show_coordinate_frame=True, show_connections=True)